# 08 · Near-duplicate check and four-class parameter counts (no training)

Two CPU-only checks that the byte-level MD5 scan of the paper cannot do.

1. **Perceptual-hash near-duplicates.** MD5 finds only byte-identical files. A 64-bit DCT perceptual hash (pHash) also finds re-encoded, resized or lightly edited copies. The notebook reports, for several Hamming-distance thresholds,
   * BRISC2025 **test** images that have a near-duplicate in the **training** pool (internal leakage),
   * near-duplicate pairs **inside** the training pool,
   * images of each **external** set that have a near-duplicate in BRISC2025, and how many of those are *not* byte-identical (the part MD5 missed).
2. **Parameter counts** of MobileNetV4-Small and RepViT-M1 (timm, the constructors used in notebooks 02-03) with a four-class head and with the library 1000-class head.

pHash detects visual similarity, not patient identity. A small distance means "very similar image", so the counts are a screen for leakage, not proof of it. Outputs go to `/kaggle/working`.

**Attach:** BRISC2025 and the external Mendeley sets (any that are missing are skipped).

In [ ]:
!pip install -q timm
import os, hashlib, math, itertools
from pathlib import Path
import numpy as np, pandas as pd, cv2
OUT_DIR = Path("/kaggle/working"); OUT_DIR.mkdir(exist_ok=True, parents=True)
IMG_EXT = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}

In [ ]:
BRISC_ROOT = Path("/kaggle/input/datasets/briscdataset/brisc2025/brisc2025/classification_task")
BRISC_TRAIN_ROOT, BRISC_TEST_ROOT = BRISC_ROOT / "train", BRISC_ROOT / "test"
assert BRISC_TRAIN_ROOT.exists() and BRISC_TEST_ROOT.exists(), "Attach BRISC2025 (classification_task) to this notebook."

# Same external roots as notebook 07. Missing ones are skipped.
_ds3 = Path("/kaggle/input/datasets/neyamul/bt-mendeley-3")
_ds3_sub = [d for d in _ds3.iterdir() if d.is_dir()] if _ds3.exists() else []
EXTERNAL = {
    "mendeley3": (_ds3_sub[0] / "Test") if len(_ds3_sub) == 1 else None,
    "mendeley5": Path("/kaggle/input/datasets/neyamul/bt-mendeley-5/Brain Tumor data/Testing"),
    "mendeley_epic_cscr": Path("/kaggle/input/datasets/neyamul/mendeley-bt-data/Epic and CSCR hospital Dataset/Test"),
    "mendeley2": Path("/kaggle/input/datasets/neyamul/mendeley-2/test"),
}

def list_images(root):
    return sorted(p for p in Path(root).rglob("*") if p.suffix.lower() in IMG_EXT)

In [ ]:
def md5_of(path, chunk=1 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

def phash64(path):
    img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None
    img = cv2.resize(img, (32, 32), interpolation=cv2.INTER_AREA).astype(np.float32)
    low = cv2.dct(img)[:8, :8].flatten()
    med = np.median(low[1:])                      # ignore the DC term
    bits = (low > med).astype(np.uint64)
    return np.uint64(sum(int(b) << i for i, b in enumerate(bits)))

def describe(paths, name):
    rows = []
    for p in paths:
        h = phash64(p)
        if h is not None:
            rows.append((str(p), md5_of(p), h))
    df = pd.DataFrame(rows, columns=["path", "md5", "phash"])
    df["set"] = name
    print(f"{name}: {len(df)} images hashed (of {len(paths)})")
    return df

POP = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)
def min_hamming(Q, R, exclude_self=False, chunk=256):
    """For every hash in Q the smallest Hamming distance to any hash in R."""
    Q = np.asarray(Q, dtype=np.uint64); R = np.asarray(R, dtype=np.uint64)
    out = np.empty(len(Q), dtype=np.int16)
    for s in range(0, len(Q), chunk):
        x = (Q[s:s + chunk, None] ^ R[None, :]).view(np.uint8).reshape(-1, len(R), 8)
        d = POP[x].sum(-1).astype(np.int16)
        if exclude_self:
            for k in range(d.shape[0]):
                d[k, s + k] = 99
        out[s:s + chunk] = d.min(1)
    return out

In [ ]:
train_df = describe(list_images(BRISC_TRAIN_ROOT), "brisc_train_pool")
test_df  = describe(list_images(BRISC_TEST_ROOT),  "brisc_test")
THRESH = [0, 2, 4, 6, 8, 10]

# (a) test images with a near-duplicate in the training pool
d_test = min_hamming(test_df["phash"].values, train_df["phash"].values)
test_df["min_dist_to_train"] = d_test
test_df["md5_in_train"] = test_df["md5"].isin(set(train_df["md5"]))
# (b) near-duplicates inside the training pool (each image against all others)
d_train = min_hamming(train_df["phash"].values, train_df["phash"].values, exclude_self=True)
train_df["min_dist_to_other_train"] = d_train

rows = []
for t in THRESH:
    rows.append({"threshold (Hamming <=)": t,
                 "test images with a train near-duplicate": int((d_test <= t).sum()),
                 "% of test": round(100 * (d_test <= t).mean(), 2),
                 "of which not byte-identical": int(((d_test <= t) & ~test_df["md5_in_train"]).sum()),
                 "train images with another train near-duplicate": int((d_train <= t).sum()),
                 "% of train": round(100 * (d_train <= t).mean(), 2)})
internal = pd.DataFrame(rows); print(internal.to_string(index=False))
internal.to_csv(OUT_DIR / "phash_internal_summary.csv", index=False)
test_df.drop(columns=["phash"]).to_csv(OUT_DIR / "phash_test_vs_train.csv", index=False)

In [ ]:
ext_rows = []
brisc_all = pd.concat([train_df, test_df], ignore_index=True)
brisc_md5 = set(brisc_all["md5"])
for name, root in EXTERNAL.items():
    if root is None or not Path(root).exists():
        print(f"[skip] {name}: not attached"); continue
    e = describe(list_images(root), name)
    e["min_dist_to_brisc"] = min_hamming(e["phash"].values, brisc_all["phash"].values)
    e["md5_in_brisc"] = e["md5"].isin(brisc_md5)
    e.drop(columns=["phash"]).to_csv(OUT_DIR / f"phash_{name}_vs_brisc.csv", index=False)
    for t in THRESH:
        near = e["min_dist_to_brisc"] <= t
        ext_rows.append({"external set": name, "n": len(e), "threshold (Hamming <=)": t,
                         "byte-identical (MD5)": int(e["md5_in_brisc"].sum()),
                         "near-duplicate (pHash)": int(near.sum()),
                         "near-duplicate but not byte-identical": int((near & ~e["md5_in_brisc"]).sum()),
                         "no near-duplicate": int((~near).sum())})
external = pd.DataFrame(ext_rows)
print(external.to_string(index=False))
external.to_csv(OUT_DIR / "phash_external_summary.csv", index=False)

In [ ]:
import timm, torch
rows = []
for label, name in [("MobileNetV4-Small", "mobilenetv4_conv_small"), ("RepViT-M1", "repvit_m1")]:
    for nc in (1000, 4):
        m = timm.create_model(name, pretrained=False, num_classes=nc)
        rows.append({"model": label, "timm name": name, "classes": nc,
                     "parameters": sum(p.numel() for p in m.parameters()),
                     "parameters (M)": round(sum(p.numel() for p in m.parameters()) / 1e6, 3)})
params = pd.DataFrame(rows); print(params.to_string(index=False))
params.to_csv(OUT_DIR / "baseline_param_counts_4class.csv", index=False)

## For the paper
* Report the row of `phash_internal_summary.csv` at the threshold you choose (a distance of 4 to 6 is a common screen; state the threshold). Say that pHash finds visually similar images, not same-patient images.
* Report `near-duplicate but not byte-identical` for each external set: that is the contamination the MD5 scan missed.
* Use `baseline_param_counts_4class.csv` for the four-class counts of Table 9 (the library 1000-class values there are 3.77M and 4.72M).